# Profile data

## Generate summary statistics using SQL

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/02-generate-summary-statistics-using-sql.svg)

In [ ]:
-- Collect basic table statistics
ANALYZE TABLE sales.transactions COMPUTE STATISTICS;

-- Collect statistics for specific columns
ANALYZE TABLE sales.transactions COMPUTE STATISTICS FOR COLUMNS 
    transaction_id, amount, customer_id;

-- View column-level statistics
DESC EXTENDED sales.transactions amount;


In [ ]:
# Generate summary statistics using PySpark
df = spark.table("sales.transactions")
df.describe().show()


## Use data profiling in Unity Catalog

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/02-use-data-profiling-in-unity-catalog.svg)

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorSnapshot

w = WorkspaceClient()
# Create a snapshot profile for a table
# w.quality_monitors.create(
#     table_name="main.sales.transactions",
#     assets_dir="/Shared/monitors/transactions",
#     snapshot=MonitorSnapshot()
# )
print("Navigate to Catalog Explorer > Table > Quality > Configure to enable data profiling")


## Interpret profiling results

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/02-interpret-profiling-results.svg)

In [ ]:
-- Count nulls per column to assess data completeness
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(amount) AS null_amount_count,
    COUNT(*) - COUNT(customer_id) AS null_customer_count
FROM sales.transactions;


# Choose column data types

## Understand data type categories

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/03-understand-data-type-categories.svg)

In [ ]:
-- Example showing all data type categories
CREATE TABLE IF NOT EXISTS demo.type_examples (
    -- Numeric integral types
    tiny_val TINYINT,        -- -128 to 127
    small_val SMALLINT,      -- -32,768 to 32,767
    int_val INT,             -- -2.1B to 2.1B
    big_val BIGINT,          -- -9.2Q to 9.2Q
    -- Decimal and floating point
    exact_price DECIMAL(10,2),
    approx_val DOUBLE,
    -- Date-time types
    event_date DATE,
    event_ts TIMESTAMP,
    scheduled_ts TIMESTAMP_NTZ,
    -- Text and binary
    description STRING,
    raw_data BINARY,
    -- Complex types
    tags ARRAY<STRING>,
    attributes MAP<STRING, STRING>,
    address STRUCT<street:STRING, city:STRING, zip:STRING>
);


## Select the right numeric type

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/03-select-right-numeric-type.svg)

In [ ]:
-- Whole number type selection
CREATE TABLE IF NOT EXISTS demo.orders_typed (
    order_id INT,             -- Up to 2.1B
    status_code TINYINT,      -- Small set of codes
    total_items SMALLINT,     -- Limited range
    transaction_id BIGINT     -- Large unique IDs
);

-- Decimal vs floating point
CREATE TABLE IF NOT EXISTS demo.financial (
    transaction_id INT,
    amount DECIMAL(10,2),     -- EXACT: use for money!
    tax_rate DECIMAL(5,4),    -- EXACT: percentage
    scientific_val DOUBLE     -- APPROXIMATE: scientific
);


## Choose temporal types correctly

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/03-choose-temporal-types-correctly.svg)

In [ ]:
-- Temporal type selection examples
CREATE TABLE IF NOT EXISTS demo.temporal_types (
    employee_id INT,
    hire_date DATE,             -- Calendar date only
    last_login TIMESTAMP,       -- Timezone-aware (UTC internally)
    scheduled_meeting TIMESTAMP_NTZ  -- No timezone conversion
);

-- Show difference between TIMESTAMP and TIMESTAMP_NTZ
SELECT 
    CAST('2026-03-15 14:30:00' AS TIMESTAMP) AS ts_tz,
    CAST('2026-03-15 14:30:00' AS TIMESTAMP_NTZ) AS ts_ntz;


## Work with complex types

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/03-work-with-complex-types.svg)

In [ ]:
-- Array type example
CREATE TABLE IF NOT EXISTS demo.products_complex (
    product_id INT,
    name STRING,
    tags ARRAY<STRING>,
    properties MAP<STRING, STRING>
);

-- Querying complex types
SELECT 
    product_id,
    tags[0] AS primary_tag,
    properties['color'] AS color
FROM demo.products_complex;


## Apply type selection best practices

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/03-apply-type-selection-best-practices.svg)

In [ ]:
-- Cast during ingestion to validate types early
INSERT INTO demo.orders_typed
SELECT 
    CAST(raw_id AS INT) AS order_id,
    CAST(raw_status AS TINYINT) AS status_code,
    CAST(raw_amount AS DECIMAL(10,2)) AS amount,
    CAST(raw_date AS DATE) AS order_date
FROM staging.raw_orders;


# Resolve duplicates and nulls

## Identify duplicate records

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/04-identify-duplicate-records.svg)

In [ ]:
-- Find duplicates using GROUP BY + HAVING
SELECT 
    customer_id, 
    email, 
    COUNT(*) AS occurrence_count
FROM sales.customers
GROUP BY customer_id, email
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;


In [ ]:
-- Find duplicate rows using QUALIFY + window function
SELECT *
FROM sales.customers
QUALIFY COUNT(*) OVER (PARTITION BY customer_id, email) > 1;


In [ ]:
# Find duplicates in PySpark using exceptAll()
df = spark.table("sales.customers")
df_duplicates = df.exceptAll(df.dropDuplicates(["customer_id", "email"]))
display(df_duplicates)


## Remove duplicate records

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/04-remove-duplicate-records.svg)

In [ ]:
# Remove duplicates in PySpark - keep first occurrence
df_clean = df.dropDuplicates(["customer_id", "email"])

# Full row deduplication 
df_unique = df.distinct()


In [ ]:
-- Keep most recently updated record using ROW_NUMBER + QUALIFY
SELECT *
FROM sales.customers
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY customer_id 
    ORDER BY updated_at DESC
) = 1;


## Handle null and missing values

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/04-handle-null-missing-values.svg)

In [ ]:
-- Identify nulls in SQL
SELECT *
FROM sales.transactions
WHERE amount IS NULL 
   OR customer_id IS NULL;


In [ ]:
# Handle nulls in PySpark
df = spark.table("sales.transactions")

# Drop rows with any null in specified columns
df_clean = df.dropna(subset=["customer_id", "amount"])

# Fill null values with defaults
df_filled = df.fillna({
    "amount": 0,
    "status": "Unknown",
    "quantity": 1
})


In [ ]:
-- Fill null values in SQL using COALESCE
SELECT 
    customer_id,
    COALESCE(amount, 0) AS amount,
    COALESCE(status, 'Unknown') AS status
FROM sales.transactions;


# Transform data with filters and aggregations

## Filter data to select relevant rows

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/05-filter-data.svg)

In [ ]:
from pyspark.sql.functions import col

df_customer = spark.table("samples.tpch.customer")

# Filter with single condition
df_filtered = df_customer.filter(col("c_acctbal") > 1000)

# Filter with multiple conditions (AND / OR)
df_filtered_multi = df_customer.filter(
    (col("c_nationkey") == 20) & (col("c_acctbal") > 1000)
)

display(df_filtered_multi)


In [ ]:
-- Filter with SQL WHERE clause
SELECT *
FROM samples.tpch.orders
WHERE o_orderstatus = 'F'
  AND o_totalprice > 50000;


## Group data to organize records

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/05-group-data.svg)

In [ ]:
# Group with PySpark - groupBy returns GroupedData object
df_grouped = df_customer.groupBy("c_mktsegment")

# Group by multiple columns
df_grouped_multi = df_customer.groupBy("c_mktsegment", "c_nationkey")


In [ ]:
-- Group with SQL - every SELECT col must be in GROUP BY or aggregate
SELECT c_mktsegment, COUNT(*) AS customer_count
FROM samples.tpch.customer
GROUP BY c_mktsegment;


## Aggregate data to calculate summary statistics

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/05-aggregate-data.svg)

In [ ]:
from pyspark.sql.functions import avg, sum, count, min, max

# Multiple aggregations per group
df_stats = spark.table("samples.tpch.orders").groupBy("o_orderpriority").agg(
    count("o_orderkey").alias("order_count"),
    sum("o_totalprice").alias("total_revenue"),
    avg("o_totalprice").alias("avg_order_value"),
    max("o_totalprice").alias("max_order")
)
display(df_stats)


In [ ]:
-- SQL aggregation with HAVING to filter groups
SELECT o_orderpriority,
       COUNT(*) AS order_count,
       SUM(o_totalprice) AS total_revenue,
       AVG(o_totalprice) AS avg_order_value
FROM samples.tpch.orders
GROUP BY o_orderpriority
HAVING COUNT(*) > 1000
ORDER BY total_revenue DESC;


## Chain transformations for readable pipelines

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/05-chain-transformations.svg)

In [ ]:
from pyspark.sql.functions import col, count

# Chain filter → groupBy → agg → sort (lazy evaluation)
df_result = (
    spark.table("samples.tpch.orders")
    .filter(col("o_orderstatus") == "F")
    .groupBy("o_orderpriority")
    .agg(count("o_orderkey").alias("n_orders"))
    .sort(col("n_orders").desc())
)
display(df_result)


# Transform data with joins and set operators

## Combine tables with joins

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/06-combine-tables-with-joins.svg)

In [ ]:
-- Inner join: only matching rows
SELECT e.id, e.name, e.deptno, d.deptname
FROM employee e
INNER JOIN department d ON e.deptno = d.deptno;


In [ ]:
-- Left join: all employees including those without a department
SELECT e.id, e.name, e.deptno, d.deptname
FROM employee e
LEFT JOIN department d ON e.deptno = d.deptno;


In [ ]:
# PySpark joins
df_employee = spark.table("employee")
df_department = spark.table("department")

# Inner join
df_inner = df_employee.join(df_department, on="deptno", how="inner")

# Left join
df_left = df_employee.join(df_department, on="deptno", how="left")

# Anti join - employees without a department
df_anti = df_employee.join(df_department, on="deptno", how="anti")

display(df_anti)


## Combine rows with set operators

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/06-combine-rows-with-set-operators.svg)

In [ ]:
-- UNION: combine rows, removes duplicates by default
SELECT customer_id, name FROM active_customers
UNION
SELECT customer_id, name FROM archived_customers;

-- UNION ALL: keeps all rows including duplicates (faster)
SELECT customer_id, name FROM active_customers
UNION ALL
SELECT customer_id, name FROM archived_customers;


In [ ]:
-- INTERSECT: rows in both queries
SELECT customer_id FROM active_customers
INTERSECT
SELECT customer_id FROM archived_customers;

-- EXCEPT: rows in first but not second
SELECT customer_id FROM active_customers
EXCEPT
SELECT customer_id FROM archived_customers;


## Choose between joins and set operators

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/06-choose-joins-or-set-operators.svg)

In [ ]:
-- Use JOINS to enrich records horizontally (add columns from related table)
SELECT o.order_id, o.amount, c.customer_name, c.region
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id;

-- Use SET OPERATORS to combine rows vertically (append rows from same schema)
SELECT order_id, amount, 'current' AS source FROM current_orders
UNION ALL
SELECT order_id, amount, 'archive' AS source FROM archived_orders;


# Transform data with denormalization and pivots

## Understand denormalization

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/07-understand-denormalization.svg)

In [ ]:
-- Create a denormalized gold layer table (flatten joins at write time)
CREATE OR REPLACE TABLE sales.orders_denormalized AS
SELECT 
    o.order_id,
    o.order_date,
    c.customer_name,
    c.region,
    p.product_name,
    p.category,
    o.quantity,
    o.total_price
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id;

-- Now queries are fast - no joins required at read time
SELECT region, SUM(total_price) AS revenue
FROM sales.orders_denormalized
GROUP BY region;


## Pivot rows into columns

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/07-pivot-rows-into-columns.svg)

In [ ]:
-- Create sample long-format sales data
CREATE OR REPLACE TEMP VIEW quarterly_sales(year, quarter, region, revenue) AS
VALUES (2024, 1, 'East', 150000), (2024, 2, 'East', 175000),
       (2024, 3, 'East', 160000), (2024, 4, 'East', 200000),
       (2024, 1, 'West', 180000), (2024, 2, 'West', 165000),
       (2024, 3, 'West', 190000), (2024, 4, 'West', 210000);

-- Pivot quarters into columns for side-by-side comparison
SELECT year, region, Q1, Q2, Q3, Q4
FROM quarterly_sales
PIVOT (
  SUM(revenue)
  FOR quarter IN (1 AS Q1, 2 AS Q2, 3 AS Q3, 4 AS Q4)
);


## Unpivot columns into rows

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/07-unpivot-columns-into-rows.svg)

In [ ]:
-- Create wide-format data
CREATE OR REPLACE TEMP VIEW regional_targets(region, jan, feb, mar, apr) AS
VALUES ('North', 50000, 55000, 52000, 58000),
       ('South', 45000, 48000, 51000, 53000);

-- Unpivot monthly columns back into rows (normalize wide to long format)
SELECT *
FROM regional_targets
UNPIVOT (
  target_amount FOR month IN (jan, feb, mar, apr)
);


## Choose the right transformation

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/07-choose-right-transformation.svg)

In [ ]:
-- Decision guide in code comments:

-- DENORMALIZE: gold layer tables, BI tools, eliminate join cost at read time
-- CREATE TABLE gold.orders_flat AS SELECT ... FROM orders JOIN customers JOIN products

-- PIVOT: analyst-facing, cross-tab reports, categories in rows → columns  
-- SELECT ... FROM data PIVOT (SUM(val) FOR col IN (...))

-- UNPIVOT: normalize wide source data, prepare for ML, long format needed
-- SELECT ... FROM data UNPIVOT (val FOR col IN (...))

print("Choose based on: downstream usage, write vs read tradeoffs, and schema requirements")


# Load data with merge, insert, and append

## Append data with INSERT INTO

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/08-append-data-insert-into.svg)

In [ ]:
-- Append new records
INSERT INTO sales.transactions (transaction_id, amount, transaction_date)
VALUES 
    ('TXN001', 150.00, '2026-01-15'),
    ('TXN002', 275.50, '2026-01-15');

-- Append from staging table (incremental extract)
INSERT INTO sales.transactions
SELECT * FROM staging.daily_transactions
WHERE transaction_date = current_date();


In [ ]:
# PySpark append modes
df_new = spark.table("staging.daily_transactions")

# Append to existing table
df_new.write.mode("append").saveAsTable("sales.transactions")

# Alternative: insertInto
df_new.write.insertInto("sales.transactions")


## Replace data with INSERT OVERWRITE

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/08-replace-data-insert-overwrite.svg)

In [ ]:
-- Full table overwrite (truncate + insert)
INSERT OVERWRITE sales.daily_summary
SELECT 
    region,
    SUM(amount) AS total_sales,
    COUNT(*) AS transaction_count
FROM sales.transactions
WHERE transaction_date = current_date()
GROUP BY region;


In [ ]:
-- Delta Lake selective overwrite using REPLACE WHERE
INSERT INTO sales.transactions
REPLACE WHERE transaction_date BETWEEN '2026-01-01' AND '2026-01-07'
SELECT * FROM staging.corrected_transactions;


In [ ]:
# PySpark overwrite
df_refreshed = spark.table("staging.daily_summary")

# Overwrite entire table
df_refreshed.write.mode("overwrite").saveAsTable("sales.daily_summary")


## Merge data for upsert operations

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/08-merge-data-upsert.svg)

In [ ]:
-- MERGE for upsert: update existing + insert new records
MERGE INTO customers AS target
USING customer_updates AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
    UPDATE SET 
        target.email = source.email,
        target.phone = source.phone,
        target.last_updated = current_timestamp()
WHEN NOT MATCHED THEN
    INSERT (customer_id, name, email, phone, created_date, last_updated)
    VALUES (source.customer_id, source.name, source.email, source.phone, 
            current_timestamp(), current_timestamp());


In [ ]:
-- MERGE with conditional logic (only update when data has changed)
MERGE INTO products AS target
USING product_feed AS source
ON target.sku = source.sku
WHEN MATCHED AND source.price <> target.price THEN
    UPDATE SET target.price = source.price, target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN
    INSERT (sku, name, price, updated_at)
    VALUES (source.sku, source.name, source.price, current_timestamp())
WHEN NOT MATCHED BY SOURCE THEN
    DELETE;


## Choose the right loading strategy

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.cleanse-transform-load-data-into-unity-catalog/08-choose-loading-strategy.svg)

In [ ]:
-- Summary: choose loading strategy based on use case

-- INSERT INTO (append): new records only, no overlap with existing
-- INSERT INTO logs SELECT * FROM staging WHERE date = today

-- INSERT OVERWRITE: full refresh, rebuilding aggregations
-- INSERT OVERWRITE summary SELECT region, SUM(amt) ...

-- MERGE: updates + inserts (upsert), CDC, dimension tables
-- MERGE INTO target USING source ON key WHEN MATCHED ... WHEN NOT MATCHED ...

-- REPLACE WHERE (Delta): selective correction of a partition/range
-- INSERT INTO table REPLACE WHERE date BETWEEN x AND y SELECT ...

print("Key rule: MERGE requires 1:1 match - deduplicate source data first!")
